# Phase 06A.01 — Blind question-taxonomy freeze
Creates prediction-blind annotation sheets, measures agreement, prepares adjudication, and freezes taxonomy only from an explicit adjudicated file.

In [1]:
import json, os, sys
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *


## Configuration

In [2]:
RUN_SCOPE='full'
VALIDATION_CSV=PROJECT_ROOT/'data/splits/phase01/validation.csv'
VALIDATION_IDS=PROJECT_ROOT/'data/splits/phase01/validation_sample_ids.json'
OUTPUT_DIR=PROJECT_ROOT/'outputs/phase06a/question_taxonomy'
ANNOTATOR_1=OUTPUT_DIR/'annotator_1.csv'; ANNOTATOR_2=OUTPUT_DIR/'annotator_2.csv'; ADJUDICATED=OUTPUT_DIR/'taxonomy_adjudicated.csv'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
require_run_scope(RUN_SCOPE); assert VALIDATION_CSV.is_file() and VALIDATION_IDS.is_file()

## Export blind annotation material
The sheet omits answers, model outputs, correctness, and error transitions.

In [3]:
val_df=pd.read_csv(VALIDATION_CSV); frozen_ids=json.loads(VALIDATION_IDS.read_text(encoding='utf-8'))
assert sorted(val_df.sample_id.astype(str))==sorted(map(str,frozen_ids)) and len(val_df)==EXPECTED_VALIDATION_ROWS
axes=['visual_required','temporal_required','traffic_knowledge_required','mixed_or_ambiguous']
blind_columns=['sample_id','group_id','video_path','question','option_a','option_b','option_c','option_d']
blind=val_df[blind_columns].copy()
for column in axes: blind[column]=''
blind['primary_label']=''; blind['annotator_note']=''
blind.to_csv(OUTPUT_DIR/'taxonomy_annotation_blind.csv',index=False)
guideline='''# RoadBuddy question taxonomy guideline

Annotate without viewing model predictions or correctness. Binary axes use 0/1.

- visual_required: a static visual observation is necessary.
- temporal_required: motion, order, direction, or change over time is necessary.
- traffic_knowledge_required: an external traffic rule/convention is necessary.
- mixed_or_ambiguous: evidence requirements cannot be assigned cleanly.
- primary_label: visual_static, temporal, traffic_knowledge, mixed, or ambiguous.

Inspect the video when the question alone is insufficient. Do not consult prediction artifacts.
'''
(OUTPUT_DIR/'ANNOTATION_GUIDELINE.md').write_text(guideline,encoding='utf-8')

595

## Agreement and adjudication gate

In [4]:
if not ANNOTATOR_1.is_file() or not ANNOTATOR_2.is_file():
    save_json(OUTPUT_DIR/'PHASE06A_01_STATUS.json',{'phase':'06A.01','status':'awaiting_annotation','required':[str(ANNOTATOR_1),str(ANNOTATOR_2)]})
    raise RuntimeError('Complete two independent annotation files before continuing')
left=pd.read_csv(ANNOTATOR_1); right=pd.read_csv(ANNOTATOR_2)
agreement=annotation_agreement(left,right); disagreements=agreement.pop('disagreements')
save_json(OUTPUT_DIR/'agreement_metrics.json',agreement)
disagreements.to_csv(OUTPUT_DIR/'adjudication_required.csv',index=False)
display(agreement)
if not ADJUDICATED.is_file():
    save_json(OUTPUT_DIR/'PHASE06A_01_STATUS.json',{'phase':'06A.01','status':'awaiting_adjudication','disagreement_rows':len(disagreements)})
    raise RuntimeError('Create taxonomy_adjudicated.csv; taxonomy is not frozen yet')

RuntimeError: Complete two independent annotation files before continuing

## Freeze adjudicated taxonomy

In [ ]:
frozen=pd.read_csv(ADJUDICATED)
required={'sample_id','primary_label',*axes}; missing=required-set(frozen.columns); assert not missing,missing
assert frozen.sample_id.astype(str).is_unique and sorted(frozen.sample_id.astype(str))==sorted(map(str,frozen_ids))
assert not frozen[list(required-{'sample_id'})].isna().any().any()
frozen.to_csv(OUTPUT_DIR/'taxonomy_frozen.csv',index=False)
taxonomy_hash=sha256_file(OUTPUT_DIR/'taxonomy_frozen.csv')
save_json(OUTPUT_DIR/'taxonomy_manifest.json',{'rows':len(frozen),'sha256':taxonomy_hash,'agreement':agreement})
save_json(OUTPUT_DIR/'PHASE06A_01_STATUS.json',{'phase':'06A.01','status':'complete','rows':len(frozen),'taxonomy_sha256':taxonomy_hash})

## Interpretation constraint
Taxonomy slices are unavailable until adjudication is complete. Agreement statistics describe annotation reliability, not model performance.